# Convert Pytorch weight to be tiny-engine compatible

In [1]:
import os
os.environ['https_proxy'] = 'http://192.168.1.22:7890'
debug = True
if debug:
    # improve torch tensor printing
    import torch
    def custom_repr(self):
        return f'{{Tensor:{tuple(self.shape)}}} {original_repr(self)}'
    original_repr = torch.Tensor.__repr__
    torch.Tensor.__repr__ = custom_repr

In [2]:
# Load model directly
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True, fp32=True).eval()

Your device support faster inference by passing bf16=True in "AutoModelForCausalLM.from_pretrained".


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [3]:
model

QWenLMHeadModel(
  (transformer): QWenModel(
    (wte): Embedding(151936, 4096)
    (drop): Dropout(p=0.0, inplace=False)
    (rotary_emb): RotaryEmbedding()
    (h): ModuleList(
      (0-31): 32 x QWenBlock(
        (ln_1): RMSNorm()
        (attn): QWenAttention(
          (c_attn): Linear(in_features=4096, out_features=12288, bias=True)
          (c_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (attn_dropout): Dropout(p=0.0, inplace=False)
        )
        (ln_2): RMSNorm()
        (mlp): QWenMLP(
          (w1): Linear(in_features=4096, out_features=11008, bias=False)
          (w2): Linear(in_features=4096, out_features=11008, bias=False)
          (c_proj): Linear(in_features=11008, out_features=4096, bias=False)
        )
      )
    )
    (ln_f): RMSNorm()
  )
  (lm_head): Linear(in_features=4096, out_features=151936, bias=False)
)

In [4]:
model.transformer.h[0]

QWenBlock(
  (ln_1): RMSNorm()
  (attn): QWenAttention(
    (c_attn): Linear(in_features=4096, out_features=12288, bias=True)
    (c_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (attn_dropout): Dropout(p=0.0, inplace=False)
  )
  (ln_2): RMSNorm()
  (mlp): QWenMLP(
    (w1): Linear(in_features=4096, out_features=11008, bias=False)
    (w2): Linear(in_features=4096, out_features=11008, bias=False)
    (c_proj): Linear(in_features=11008, out_features=4096, bias=False)
  )
)

In [5]:
output_path = "qwen-7b-chat"
os.makedirs(output_path, exist_ok=True)
with torch.no_grad():
    with open(os.path.join(output_path, "lm_head.bin"), "wb") as f:
        f.write(model.lm_head._parameters["weight"].cpu().float().numpy().tobytes())


In [6]:
OUTPUT_PATH = dict()
output_path = os.path.join("qwen-7b-chat", "transformer")

# export wte
OUTPUT_PATH.update(
    wte_weight = os.path.join(output_path, "wte", "weight.bin"),
    ln_f_weight = os.path.join(output_path, "norm", "weight.bin")
)
for path in OUTPUT_PATH.values():
    dirname = os.path.dirname(path)
    os.makedirs(dirname, exist_ok=True)

print(OUTPUT_PATH)
with torch.no_grad():
    with open(OUTPUT_PATH['wte_weight'], "wb") as f:
        f.write(model.transformer.wte.weight.cpu().float().numpy().tobytes())
    with open(OUTPUT_PATH['ln_f_weight'], "wb") as f:
        f.write(model.transformer.ln_f.weight.cpu().float().numpy().tobytes())
    

{'wte_weight': 'qwen-7b-chat/transformer/wte/weight.bin', 'ln_f_weight': 'qwen-7b-chat/transformer/norm/weight.bin'}


In [ ]:
QWEN_BLOCKS = model.transformer.h
for idx, layer in enumerate(QWEN_BLOCKS):
    OUTPUT_PATH = dict()
    output_path = os.path.join("qwen-7b-chat", "transformer", f"layer{idx}")

    # export wte
    OUTPUT_PATH.update(
        ln_1_weight   = os.path.join(output_path, "ln_1", "weight.bin"),
        ln_2_weight   = os.path.join(output_path, "ln_2", "weight.bin"),
        w1_weight     = os.path.join(output_path, "w1", "weight.bin"),
        w2_weight     = os.path.join(output_path, "w2", "weight.bin"),
        c_proj_weight = os.path.join(output_path, "c_proj", "weight.bin"),
        q_proj_weight = os.path.join(output_path, "self_attn", "q_proj", "weight.bin"),
        q_proj_bias   = os.path.join(output_path, "self_attn", "q_proj", "bias.bin"),
        k_proj_weight = os.path.join(output_path, "self_attn", "k_proj", "weight.bin"),
        k_proj_bias   = os.path.join(output_path, "self_attn", "k_proj", "bias.bin"),
        v_proj_weight = os.path.join(output_path, "self_attn", "v_proj", "weight.bin"),
        v_proj_bias   = os.path.join(output_path, "self_attn", "v_proj", "bias.bin"),
        o_proj_weight = os.path.join(output_path, "self_attn", "o_proj", "weight.bin"),

    )
    for path in OUTPUT_PATH.values():
        dirname = os.path.dirname(path)
        os.makedirs(dirname, exist_ok=True)

    # split the origianl projection into qkv
    q_proj_weight, k_proj_weight, v_proj_weight = torch.split(layer.attn.c_attn.weight.data, 4096, dim=0)
    q_proj_bias, k_proj_bias, v_proj_bias = torch.split(layer.attn.c_attn.bias.data, 4096, dim=0)
    o_proj_weight = layer.attn.c_proj.weight
    
    print(OUTPUT_PATH)
    with torch.no_grad():
        with open(OUTPUT_PATH['ln_1_weight'], "wb") as f:
            f.write(layer.ln_1.weight.cpu().float().numpy().tobytes())
        with open(OUTPUT_PATH['ln_2_weight'], "wb") as f:
            f.write(layer.ln_2.weight.cpu().float().numpy().tobytes())
        with open(OUTPUT_PATH['w1_weight'], "wb") as f:
            f.write(layer.mlp.w1.weight.cpu().float().numpy().tobytes())
        with open(OUTPUT_PATH['w2_weight'], "wb") as f:
            f.write(layer.mlp.w2.weight.cpu().float().numpy().tobytes())
        with open(OUTPUT_PATH['c_proj_weight'], "wb") as f:
            f.write(layer.mlp.c_proj.weight.cpu().float().numpy().tobytes())
        with open(OUTPUT_PATH['q_proj_weight'], "wb") as f:
            f.write(q_proj_weight.cpu().float().numpy().tobytes())
        with open(OUTPUT_PATH['q_proj_bias'], "wb") as f:
            f.write(q_proj_bias.cpu().float().numpy().tobytes())
        with open(OUTPUT_PATH['k_proj_weight'], "wb") as f:
            f.write(k_proj_weight.cpu().float().numpy().tobytes())
        with open(OUTPUT_PATH['k_proj_bias'], "wb") as f:
            f.write(k_proj_bias.cpu().float().numpy().tobytes())
        with open(OUTPUT_PATH['v_proj_weight'], "wb") as f:
            f.write(v_proj_weight.cpu().float().numpy().tobytes())
        with open(OUTPUT_PATH['v_proj_bias'], "wb") as f:
            f.write(v_proj_bias.cpu().float().numpy().tobytes())
        with open(OUTPUT_PATH['o_proj_weight'], "wb") as f:
            f.write(o_proj_weight.cpu().float().numpy().tobytes())
    

{'ln_1_weight': 'qwen-7b-chat/transformer/layer0/ln_1/weight.bin', 'ln_2_weight': 'qwen-7b-chat/transformer/layer0/ln_2/weight.bin', 'w1_weight': 'qwen-7b-chat/transformer/layer0/w1/weight.bin', 'w2_weight': 'qwen-7b-chat/transformer/layer0/w2/weight.bin', 'c_proj_weight': 'qwen-7b-chat/transformer/layer0/c_proj/weight.bin', 'q_proj_weight': 'qwen-7b-chat/transformer/layer0/self_attn/q_proj/weight.bin', 'q_proj_bias': 'qwen-7b-chat/transformer/layer0/self_attn/q_proj/bias.bin', 'k_proj_weight': 'qwen-7b-chat/transformer/layer0/self_attn/k_proj/weight.bin', 'k_proj_bias': 'qwen-7b-chat/transformer/layer0/self_attn/k_proj/bias.bin', 'v_proj_weight': 'qwen-7b-chat/transformer/layer0/self_attn/v_proj/weight.bin', 'v_proj_bias': 'qwen-7b-chat/transformer/layer0/self_attn/v_proj/bias.bin', 'o_proj_weight': 'qwen-7b-chat/transformer/layer0/self_attn/o_proj/weight.bin'}


In [8]:
qwen_block = model.transformer.h[0]
list(qwen_block.attn.named_parameters())
torch.split( qwen_block.attn.c_attn.weight, 4096, dim=0)
torch.split( qwen_block.attn.c_attn.bias, 4096, dim=0)

({Tensor:(4096,)} tensor([-0.9453,  1.8828, -0.7461,  ...,  0.0091,  0.3711,  0.4297],
        grad_fn=<SplitBackward0>),
 {Tensor:(4096,)} tensor([-2.5781,  0.9375, -2.2969,  ..., -0.4707,  0.1245, -0.4375],
        grad_fn=<SplitBackward0>),
 {Tensor:(4096,)} tensor([ 0.0069, -0.0283, -0.0008,  ...,  0.0002, -0.0031, -0.0011],
        grad_fn=<SplitBackward0>))

In [18]:
model.transformer.rotary_emb.inv_freq

{Tensor:(64,)} tensor([1.0000e+00, 8.6596e-01, 7.4989e-01, 6.4938e-01, 5.6234e-01, 4.8697e-01,
        4.2170e-01, 3.6517e-01, 3.1623e-01, 2.7384e-01, 2.3714e-01, 2.0535e-01,
        1.7783e-01, 1.5399e-01, 1.3335e-01, 1.1548e-01, 1.0000e-01, 8.6596e-02,
        7.4989e-02, 6.4938e-02, 5.6234e-02, 4.8697e-02, 4.2170e-02, 3.6517e-02,
        3.1623e-02, 2.7384e-02, 2.3714e-02, 2.0535e-02, 1.7783e-02, 1.5399e-02,
        1.3335e-02, 1.1548e-02, 1.0000e-02, 8.6596e-03, 7.4989e-03, 6.4938e-03,
        5.6234e-03, 4.8697e-03, 4.2170e-03, 3.6517e-03, 3.1623e-03, 2.7384e-03,
        2.3714e-03, 2.0535e-03, 1.7783e-03, 1.5399e-03, 1.3335e-03, 1.1548e-03,
        1.0000e-03, 8.6596e-04, 7.4989e-04, 6.4938e-04, 5.6234e-04, 4.8697e-04,
        4.2170e-04, 3.6517e-04, 3.1623e-04, 2.7384e-04, 2.3714e-04, 2.0535e-04,
        1.7783e-04, 1.5399e-04, 1.3335e-04, 1.1548e-04])

In [8]:
base = 10000
dim =128
# inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
rope_embed = model.transformer.rotary_emb
rope_embed.update_rotary_pos_emb_cache(1024)
embed_cache = rope_embed._rotary_pos_emb_cache
cos, sin = embed_cache
cos



{Tensor:(1, 2048, 1, 128)} tensor([[[[ 1.0000,  1.0000,  1.0000,  ...,  1.0000,  1.0000,  1.0000]],

         [[ 0.5403,  0.6479,  0.7318,  ...,  1.0000,  1.0000,  1.0000]],

         [[-0.4161, -0.1604,  0.0709,  ...,  1.0000,  1.0000,  1.0000]],

         ...,

         [[-0.9844,  0.5726,  0.9062,  ...,  0.9508,  0.9630,  0.9722]],

         [[-0.6799,  0.9955,  0.3750,  ...,  0.9508,  0.9630,  0.9722]],

         [[ 0.2497,  0.7174, -0.3574,  ...,  0.9507,  0.9630,  0.9722]]]])

In [9]:
OUTPUT_PATH = dict()
output_path = os.path.join("qwen-7b-chat", "transformer")

# export wte
OUTPUT_PATH.update(
    sin_cache = os.path.join(output_path, "rotary_emb", "sin_cached.bin"),
    cos_cache = os.path.join(output_path, "rotary_emb", "cos_cached.bin")
)
for path in OUTPUT_PATH.values():
    dirname = os.path.dirname(path)
    os.makedirs(dirname, exist_ok=True)

print(OUTPUT_PATH)
with torch.no_grad():
    with open(OUTPUT_PATH['sin_cache'], "wb") as f:
        f.write(sin.cpu().float().numpy().tobytes())
    with open(OUTPUT_PATH['cos_cache'], "wb") as f:
        f.write(cos.cpu().float().numpy().tobytes())
    

{'sin_cache': 'qwen-7b-chat/transformer/rotary_emb/sin_cached.bin', 'cos_cache': 'qwen-7b-chat/transformer/rotary_emb/cos_cached.bin'}


## scaling factor of atten weight

In [10]:
import math
import struct

QWEN_BLOCKS = model.transformer.h
for idx, layer in enumerate(QWEN_BLOCKS):
    OUTPUT_PATH = dict()
    output_path = os.path.join("qwen-7b-chat", "transformer", f"layer{idx}")

    # export wte
    OUTPUT_PATH.update(
        qk_bmm_alpha = os.path.join(output_path, "self_attn/qk_bmm/alpha.bin"),
    )
    for path in OUTPUT_PATH.values():
        dirname = os.path.dirname(path)
        os.makedirs(dirname, exist_ok=True)

    alpha = 1 / math.sqrt(layer.attn.head_dim)
    
    print(OUTPUT_PATH)
    with torch.no_grad():
        with open(OUTPUT_PATH['qk_bmm_alpha'], "wb") as f:
            f.write(struct.pack("f", alpha))

{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer0/self_attn/qk_bmm/alpha.bin'}
{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer1/self_attn/qk_bmm/alpha.bin'}
{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer2/self_attn/qk_bmm/alpha.bin'}
{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer3/self_attn/qk_bmm/alpha.bin'}
{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer4/self_attn/qk_bmm/alpha.bin'}
{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer5/self_attn/qk_bmm/alpha.bin'}
{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer6/self_attn/qk_bmm/alpha.bin'}
{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer7/self_attn/qk_bmm/alpha.bin'}
{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer8/self_attn/qk_bmm/alpha.bin'}
{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer9/self_attn/qk_bmm/alpha.bin'}
{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer10/self_attn/qk_bmm/alpha.bin'}
{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer11/self_attn/qk_bmm/alpha.bin'}
{'qk_bmm_alpha': 'qwen-7b-chat/transformer/layer12

In [11]:
alpha

0.08838834764831843

# Dump activation and input to validate cpp implementation



## Attention module

### register input and output hook

In [1]:
inputs_dict = []
outputs_dict = []
def hook_func(module, input, output):
    # print("=module=")
    # print(module)
    # print("=input=")
    # print(input)
    # print("=output=")
    # print(output)
    inputs_dict.append(input)
    outputs_dict.append(output)

hook = model.transformer.h[0].attn.register_forward_hook(hook_func)
hook = model.transformer.h[0].attn.register_forward_hook(hook_func)

NameError: name 'model' is not defined

In [9]:
# hook.remove()

In [1]:
import os
os.environ['https_proxy'] = 'http://192.168.1.22:7890'
debug = True
if debug:
    # improve torch tensor printing
    import torch
    def custom_repr(self):
        return f'{{Tensor:{tuple(self.shape)}}} {original_repr(self)}'
    original_repr = torch.Tensor.__repr__
    torch.Tensor.__repr__ = custom_repr

In [2]:
# Load model directly
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True, fp32=True).eval()

Your device support faster inference by passing bf16=True in "AutoModelForCausalLM.from_pretrained".


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.generation import GenerationConfig

# Note: The default behavior now has injection attack prevention off.
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True)

# use bf16
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="auto", trust_remote_code=True, bf16=True).eval()
# use fp16
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="auto", trust_remote_code=True, fp16=True).eval()
# use cpu only
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="cpu", trust_remote_code=True).eval()
# use auto mode, automatically select precision based on the device.
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="auto", trust_remote_code=True).eval()

# Specify hyperparameters for generation. But if you use transformers>=4.32.0, there is no need to do this.
# model.generation_config = GenerationConfig.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True) # 可指定不同的生成长度、top_p等相关超参

# 第一轮对话 1st dialogue turn
response, history = model.chat(tokenizer, "你好", history=None)
print(response)

/root/miniconda3/envs/tinyml/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1614: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [5]:
input = inputs_dict[0][0]
output_attn = outputs_dict[0][0]
input, output_attn

({Tensor:(1, 20, 4096)} tensor([[[ 0.0099, -0.0241,  0.0344,  ..., -0.0008, -0.0185, -0.0072],
          [-0.1132, -0.1359, -0.0490,  ..., -0.0123,  0.0712,  0.0126],
          [-0.0245,  0.0012, -0.0050,  ...,  0.0014,  0.0277, -0.0156],
          ...,
          [ 0.0099, -0.0241,  0.0344,  ..., -0.0008, -0.0185, -0.0072],
          [-0.0664, -0.1269, -0.0096,  ..., -0.0835, -0.0816, -0.1260],
          [-0.0245,  0.0012, -0.0050,  ...,  0.0014,  0.0277, -0.0156]]]),
 {Tensor:(1, 20, 4096)} tensor([[[-0.0776,  0.0330, -0.0400,  ..., -0.0069, -0.0060, -0.0161],
          [-0.0454, -0.0116, -0.1370,  ..., -0.0124,  0.0286,  0.0095],
          [-0.0014, -0.0037, -0.0754,  ..., -0.0416,  0.0301, -0.0044],
          ...,
          [-0.0511, -0.0104, -0.0009,  ...,  0.0211,  0.0202, -0.0037],
          [-0.0931,  0.1034, -0.0824,  ..., -0.0035,  0.0635,  0.0278],
          [-0.0518,  0.0270, -0.0484,  ...,  0.0714,  0.0171,  0.0130]]]))

In [6]:
import math
import struct

OUTPUT_PATH = dict()
output_path = os.path.join("qwen-7b-chat", "transformer", f"layer0")

# export wte
OUTPUT_PATH.update(
    input_attn = os.path.join(output_path, "self_attn/activation/input_attn.bin"),
    output_attn = os.path.join(output_path, "self_attn/activation/output_attn.bin"),
)
for path in OUTPUT_PATH.values():
    dirname = os.path.dirname(path)
    os.makedirs(dirname, exist_ok=True)

print(OUTPUT_PATH)
with torch.no_grad():
    with open(OUTPUT_PATH['input_attn'], "wb") as f:
        f.write(input.cpu().numpy().tobytes())
    with open(OUTPUT_PATH['output_attn'], "wb") as f:
        f.write(output_attn.cpu().numpy().tobytes())

{'input_attn': 'qwen-7b-chat/transformer/layer0/self_attn/activation/input_attn.bin', 'output_attn': 'qwen-7b-chat/transformer/layer0/self_attn/activation/output_attn.bin'}


In [20]:
len(inputs_dict)

9

## QwenBlock

In [1]:
import os
os.environ['https_proxy'] = 'http://192.168.1.22:7890'
debug = True
if debug:
    # improve torch tensor printing
    import torch
    def custom_repr(self):
        return f'{{Tensor:{tuple(self.shape)}}} {original_repr(self)}'
    original_repr = torch.Tensor.__repr__
    torch.Tensor.__repr__ = custom_repr

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.generation import GenerationConfig

# Note: The default behavior now has injection attack prevention off.
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True, fp32=True).eval()

/root/miniconda3/envs/tinyml/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1614: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Your device support faster inference by passing bf16=True in "AutoModelForCausalLM.from_pretrained".


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [2]:

# use bf16
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="auto", trust_remote_code=True, bf16=True).eval()
# use fp16
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="auto", trust_remote_code=True, fp16=True).eval()
# use cpu only
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="cpu", trust_remote_code=True).eval()
# use auto mode, automatically select precision based on the device.
# model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen-7B-Chat", device_map="auto", trust_remote_code=True).eval()

# Specify hyperparameters for generation. But if you use transformers>=4.32.0, there is no need to do this.
# model.generation_config = GenerationConfig.from_pretrained("Qwen/Qwen-7B-Chat", trust_remote_code=True) # 可指定不同的生成长度、top_p等相关超参

# hook.remove()
inputs_dict_qwen_block = []
outputs_dict_qwen_block = []
def hook_func(module, input, output):
    # print("=module=")
    # print(module)
    # print("=input=")
    # print(input)
    # print("=output=")
    # print(output)
    inputs_dict_qwen_block.append(input)
    outputs_dict_qwen_block.append(output)

hook = model.transformer.h[0].register_forward_hook(hook_func)



# 第一轮对话 1st dialogue turn
response, history = model.chat(tokenizer, "你好", history=None)
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


你好！有什么我能帮助你的吗？


In [5]:
input_block = inputs_dict_qwen_block[0][0]
output_block = outputs_dict_qwen_block[0][0]
input_block, output_block


({Tensor:(1, 20, 4096)} tensor([[[ 1.0490e-04, -2.8229e-04,  3.3951e-04,  ..., -8.7023e-06,
           -2.0027e-04, -7.9155e-05],
          [-1.9409e-02, -2.5757e-02, -7.8125e-03,  ..., -2.1973e-03,
            1.2451e-02,  2.2430e-03],
          [-4.4250e-03,  2.3651e-04, -8.3542e-04,  ...,  2.6894e-04,
            5.0964e-03, -2.9297e-03],
          ...,
          [ 1.0490e-04, -2.8229e-04,  3.3951e-04,  ..., -8.7023e-06,
           -2.0027e-04, -7.9155e-05],
          [-9.8877e-03, -2.0874e-02, -1.3351e-03,  ..., -1.2939e-02,
           -1.2390e-02, -1.9531e-02],
          [-4.4250e-03,  2.3651e-04, -8.3542e-04,  ...,  2.6894e-04,
            5.0964e-03, -2.9297e-03]]]),
 {Tensor:(1, 20, 4096)} tensor([[[-2.2915e-01,  1.4635e-01, -1.3796e-01,  ..., -1.1090e-01,
            3.9673e-03, -8.9641e-02],
          [-8.8610e-02, -1.9821e-02, -4.1045e-02,  ..., -4.1199e-02,
            6.7500e-02,  6.0080e-02],
          [ 9.2342e-02, -1.4332e-01, -1.1347e-01,  ..., -1.5313e-01,
           

In [6]:
import math
import struct

OUTPUT_PATH = dict()
output_path = os.path.join("qwen-7b-chat", "transformer", f"layer0")

# export wte
OUTPUT_PATH.update(
    input_block = os.path.join(output_path, "activation/input.bin"),
    output_block = os.path.join(output_path, "activation/output.bin"),
)
for path in OUTPUT_PATH.values():
    dirname = os.path.dirname(path)
    os.makedirs(dirname, exist_ok=True)

print(OUTPUT_PATH)
with torch.no_grad():
    with open(OUTPUT_PATH['input_block'], "wb") as f:
        f.write(input_block.cpu().numpy().tobytes())
    with open(OUTPUT_PATH['output_block'], "wb") as f:
        f.write(output_block.cpu().numpy().tobytes())

{'input_block': 'qwen-7b-chat/transformer/layer0/activation/input.bin', 'output_block': 'qwen-7b-chat/transformer/layer0/activation/output.bin'}
